# Initiation step

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#Reading from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Silver Transformations

##Exploring the data to decide on transformations

After analyzing the data, by exploring it with df.limit(10).display() we can see we have to:
- Trim the String values (Rule for all projects)
- Do normalization on cst_marital_status and cst_gndr
- Make column name more user friendly 

Now we can start the transformation we have identify

##Trimming

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

##Renaming Columns

We are doing a mapping, a translation, from the raw data to a more user fiendly name per column. If we want to be more advance we can specify in the dictionary (in this case RENAME_MAP) a lot of things (e.g the data type, the comments, the desscription of the comments, etc.)

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity Checks of Datframe

In [0]:

df.limit(10).display()

#Write Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

##Sanity Checks of Silver Table

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_customers
LIMIT 10